In [8]:
# Install the Hugging Face hub
!pip install huggingface_hub


In [1]:
import os

print("--- DOWNLOADING SAM 2 WEIGHTS ---")

# Ensure we are in the main /content folder
%cd /content

# Download the Large model (Hiera Large) directly from Meta
if not os.path.exists("sam2_hiera_large.pt"):
    print("Downloading sam2_hiera_large.pt...")
    !wget -q https://dl.fbaipublicfiles.com/segment_anything_2/072824/sam2_hiera_large.pt
    print("✅ Download complete!")
else:
    print("✅ Model already exists.")

# Define the path for your inference script
CHECKPOINT_PATH = "/content/sam2_hiera_large.pt"
print(f"Path ready: {CHECKPOINT_PATH}")

--- DOWNLOADING SAM 2 WEIGHTS ---
/content
✅ Download complete!
Path ready: /content/sam2_hiera_large.pt


In [2]:
import torch
import sys

if torch.cuda.is_available():
    device_name = torch.cuda.get_device_name(0)
    print("✅ SUCCESS: Connected to Colab GPU")
    print(f"🤖 GPU Model: {device_name}")
    print(f"💾 VRAM Available: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("❌ STILL ON CPU: Check the top-right Kernel setting!")

# Double check we are in the cloud environment
print(f"🌐 Platform: {sys.platform}")

✅ SUCCESS: Connected to Colab GPU
🤖 GPU Model: NVIDIA RTX PRO 6000 Blackwell Server Edition
💾 VRAM Available: 101.97 GB
🌐 Platform: linux


In [4]:
import os

print("--- FIXING MISSING FILES ---")
%cd /content

# 1. Clone your homework repo if it's missing
if not os.path.exists("/content/ME_592_robotics_HW"):
    print("Cloning your GitHub Repo...")
    !git clone https://github.com/JoonParrrk/ME_592_robotics_HW.git
else:
    print("✅ GitHub Repo found.")

# 2. Download the Vocab directly to /content/ so we never lose it
if not os.path.exists("/content/bpe_simple_vocab_16e6.txt.gz"):
    print("Downloading Vocab Dictionary...")
    !wget -qO /content/bpe_simple_vocab_16e6.txt.gz https://raw.githubusercontent.com/facebookresearch/segment-anything-2/main/sam2/modeling/sam2_utils/bpe_simple_vocab_16e6.txt.gz
else:
    print("✅ Vocab found.")

print("\n🚀 ALL GAPS FILLED. You can run the main script now!")

--- FIXING MISSING FILES ---
/content
Cloning your GitHub Repo...
Cloning into 'ME_592_robotics_HW'...
remote: Enumerating objects: 2545, done.
remote: Counting objects: 100% (46/46), done.
remote: Compressing objects: 100% (42/42), done.
remote: Total 2545 (delta 7), reused 43 (delta 4), pack-reused 2499 (from 1)
Receiving objects: 100% (2545/2545), 509.15 MiB | 52.77 MiB/s, done.
Resolving deltas: 100% (430/430), done.

🚀 ALL GAPS FILLED. You can run the main script now!


In [7]:
import os

print("--- SEARCHING FOR YOUR IMAGE ---")

def find_file(filename, search_path="/content"):
    for root, dirs, files in os.walk(search_path):
        if filename in files:
            return os.path.join(root, filename)
    return None

# Look for your specific image
real_image_path = find_file("pcd0802r.png")

if real_image_path:
    print(f"✅ FOUND IT! The real path is:\n{real_image_path}")
    print("\n👉 Copy that path and paste it as your IMG_PATH in the main script!")
else:
    print("❌ The image 'pcd0802r.png' is NOT anywhere on this server.")
    print("Check your GitHub repo online—did the '6_test_images' folder actually push successfully?")

--- SEARCHING FOR YOUR IMAGE ---
✅ FOUND IT! The real path is:
/content/ME_592_robotics_HW/6_test_images/pcd0802r.png

👉 Copy that path and paste it as your IMG_PATH in the main script!


In [ ]:
# ==========================================
# 1. LOCK IN THE ABSOLUTE PATHS
# ==========================================
BASE_DIR = "/content/ME_592_robotics_HW"

# Note: We need to search inside the specific homework folder
# Check if it's named 'robotic-grasping' or something else in your repo
PROJECT_ROOT = os.path.join(BASE_DIR, "robotic-grasping") 

if BASE_DIR not in sys.path:
    sys.path.append(BASE_DIR)

# The Bulletproof Paths
CHECKPOINT_PATH = "/content/sam2_hiera_large.pt" 
VOCAB_PATH = "/content/bpe_simple_vocab_16e6.txt.gz" # <-- Updated!
IMG_PATH = os.path.join(PROJECT_ROOT, "/content/ME_592_robotics_HW/6_test_images/pcd0802r.png")

print("--- FINAL SYSTEM CHECK ---")
print(f"Vocab Ready: {os.path.exists(VOCAB_PATH)}")
print(f"Model Ready: {os.path.exists(CHECKPOINT_PATH)}")
print(f"Image Ready: {os.path.exists(IMG_PATH)}")

--- FINAL SYSTEM CHECK ---
Vocab Ready: True
Model Ready: True
Image Ready: False


In [5]:
import torch
import matplotlib.pyplot as plt
from PIL import Image
import numpy as np
import os
import sys

BASE_DIR = "/content/ME_592_robotics_HW"
PROJECT_ROOT = os.path.join(BASE_DIR, "robotic-grasping")

# Add the repo to the system path so Python can find your 'sam3' folder
if BASE_DIR not in sys.path:
    sys.path.append(BASE_DIR)

# The exact files we verified
CHECKPOINT_PATH = "/content/sam2_hiera_large.pt" 
VOCAB_PATH = os.path.join(BASE_DIR, "bpe_simple_vocab_16e6.txt.gz")
IMG_PATH = os.path.join(PROJECT_ROOT, "6_test_images/pcd0802r.png")

print("--- FINAL SYSTEM CHECK ---")
print(f"Vocab Ready: {os.path.exists(VOCAB_PATH)}")
print(f"Model Ready: {os.path.exists(CHECKPOINT_PATH)}")
print(f"Image Ready: {os.path.exists(IMG_PATH)}")

# ==========================================
# 2. LOAD THE MODEL ONTO THE BLACKWELL GPU
# ==========================================
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"\nLoading model onto {device}...")

from sam3.model_builder import build_sam3_image_model
from sam3.model.sam3_image_processor import Sam3Processor

# Note: We are feeding the SAM 2 weights into your homework's SAM 3 wrapper
model = build_sam3_image_model(
    checkpoint_path=CHECKPOINT_PATH,
    bpe_path=VOCAB_PATH
).to(device)

processor = Sam3Processor(model)

# ==========================================
# 3. RUN INFERENCE (TEXT-TO-MASK)
# ==========================================
image_pil = Image.open(IMG_PATH).convert("RGB")
inference_state = processor.set_image(image_pil)

text_prompt = "tray"
print(f"Searching image for: '{text_prompt}'...")

output = processor.set_text_prompt(state=inference_state, prompt=text_prompt)

masks = output["masks"]
boxes = output["boxes"]
scores = output["scores"]

# ==========================================
# 4. VISUALIZE ROBOTIC GRASPING COORDINATES
# ==========================================
plt.figure(figsize=(10, 10))
plt.imshow(image_pil)

if len(masks) > 0:
    # Pull from VRAM to System RAM, remove extra dimensions
    mask = masks[0].cpu().numpy().squeeze() 
    box = boxes[0].cpu().numpy()
    
    # Overlay the mask heatmap
    plt.imshow(mask, alpha=0.4, cmap='jet')
    
    # Draw Bounding Box
    x_min, y_min, x_max, y_max = box
    rect = plt.Rectangle((x_min, y_min), x_max - x_min, y_max - y_min, 
                         edgecolor='red', facecolor='none', lw=3)
    plt.gca().add_patch(rect)
    
    # Calculate and plot the center Grasping Point
    center_x, center_y = (x_min + x_max) / 2, (y_min + y_max) / 2
    plt.plot(center_x, center_y, 'x', color='yellow', markersize=15, markeredgewidth=3)
    
    plt.title(f"Target: '{text_prompt}' (Confidence: {scores[0]:.2f})", fontsize=16)
    
    print("\n" + "="*45)
    print("🤖 ROBOTIC GRASPING COORDINATES EXTRACTED:")
    print(f"Bounding Box: [{x_min:.1f}, {y_min:.1f}, {x_max:.1f}, {y_max:.1f}]")
    print(f"Target Center (X, Y): ({center_x:.1f}, {center_y:.1f})")
    print("="*45 + "\n")

else:
    plt.title(f"Model couldn't find '{text_prompt}'")
    print(f"❓ No object detected for prompt: {text_prompt}")

plt.axis('off')
plt.show()

--- FINAL SYSTEM CHECK ---
Vocab Ready: True
Model Ready: True
Image Ready: False

Loading model onto cuda...


ModuleNotFoundError: No module named 'sam3'